# Demo 1 · Anatomy of a decision call

**Part 01 · Jev and the decision layer** · about 8 minutes

We build one state, ask a **Choice**, a **Score** and a **Noul** question in a single request,
read the raw response, and recompute confidence and score ourselves so nothing in the answer is magic.

* **Live mode** (`TYPESAFE_API_KEY` set): new calls, cached in `outputs/jev_cache.jsonl`.
* **Replay mode**: real saved answers from the recorded runs, one of each type.

In [ ]:
# Setup: find the workshop folder and import the helpers (works locally and in Colab)
import os, sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/decision-models-workshop")]
             if (p / "dmw").is_dir()), None)
assert ROOT, "Open this notebook from the workshop folder. In Colab, run 00_setup.ipynb first."
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from dmw import config
config.load_env()

In [ ]:
import json
from dmw import jev

MODE = config.mode("auto")      # "auto", "live" or "replay"
print("MODE =", MODE)
show = lambda x: print(json.dumps(x, indent=2, ensure_ascii=False))

## 1 · State: three shapes

The state is the evidence. A string is one piece of text, an array is a sequence of messages,
and an object gives each fact a name. Named fields let the question refer to them directly.

In [ ]:
state_string = "Checkout returns 500 errors. Customers cannot pay."
state_array = ["Checkout returns 500 errors.", "Card payments worked yesterday."]
state_object = {
    "message": "Checkout returns 500 errors. Customers cannot pay.",
    "service_status": "unresolved incident",
    "now_utc": "2026-10-08T09:00:00Z",
    "allowed_actions": ["read", "draft"],
}
show(state_object)

## 2 · Three questions in one request

The question ID (`team`, `severity`, `blocked`) is how your code reads the answer back.

In [ ]:
QUESTIONS = {
    "team": jev.question("choice", "Which team handles `message`?", {
        "engineering": "Broken product behavior, failures or errors",
        "billing": "Invoice amounts, charges or refunds",
        "other": "No category fits, or the input is ambiguous",
    }),
    "severity": jev.question("score", "How severe is the issue in `message` for customers right now?", [
        "Cosmetic issue, no functional impact",
        "Degraded feature, a workaround exists",
        "Blocking issue, no workaround",
    ]),
    "blocked": jev.question("noul", "Does `message` report a currently blocked customer workflow?", {
        "true": "A customer cannot complete a task they are trying to do now.",
        "false": "A future plan, general information, or an issue that is already resolved.",
    }),
}
show(jev.payload(state_object, QUESTIONS))

## 3 · Call Jev and read the raw response

In [ ]:
if MODE == "live":
    out = jev.call(jev.payload(state_object, QUESTIONS))
    raw = out["response"]["answers"]
    print(f"latency {out['latency_ms']} ms  (cached: {out['cached']})")
else:
    # Replay: one real saved answer of each type from the recorded runs
    raw = {
        "team":     jev.replay_answer("gating", "missing_permission")["answers"]["decision"],
        "severity": jev.replay_answer("reranking", None, passage="reset_d", query="reset")["answers"]["relevance"],
        "blocked":  jev.replay_answer("guardrails", "doc_override")["answers"]["is_injection"],
    }
    print("Replay: a Choice from tool gating (the case Jev found hard), a Score from reranking, a Noul from guardrails.")
show(raw)

## 4 · Recompute what the response says

* **Choice confidence** = (p_max − 1/n) / (1 − 1/n). It is 0 for a uniform distribution and 1 for a certain one.
* **Score** = Σ level × probability, the probability-weighted level.
* **Noul** is P(yes) directly; P(no) = 1 − noul. There is no separate confidence field.

In [ ]:
c = raw["team"]
print("Choice:", c["choice"], "| returned confidence:", c["confidence"],
      "| recomputed:", round(jev.choice_confidence(c["probabilities"]), 3))

s = raw["severity"]
print("Score:", s["score"], "| recomputed expected level:", round(jev.expected_score(s["probabilities"]), 3))

n = raw["blocked"]
print("Noul P(yes):", n["noul"], "| P(no):", round(1 - n["noul"], 3))

## 5 · A vague message gives a flat distribution

“It does not work. Please help.” has no evidence for any team. A label is still returned,
so the distribution and confidence are what tell code to ask a follow-up question.

In [ ]:
vague = "It does not work. Please help."
if MODE == "live":
    a = jev.ask({"message": vague}, {"team": QUESTIONS["team"]})["answers"]["team"]
    print(a)
else:
    saved = jev.replay_answer("labeling", "t11")["answers"]["label"]   # same text in the bulk-labeling run
    print("Saved bulk-labeling answer for:", repr(vague)); show(saved)
    print("recomputed confidence:", round(jev.choice_confidence(saved["probabilities"]), 3))

## 6 · Change one criterion, watch the probabilities move (live only)

Remove the `other` option. With no fallback, the model must spread probability over two teams that
do not fit. This is why every Choice needs a fallback option.

In [ ]:
if MODE == "live":
    no_fallback = {"team": jev.question("choice", "Which team handles `message`?", {
        "engineering": "Broken product behavior, failures or errors",
        "billing": "Invoice amounts, charges or refunds",
    })}
    print("With other:   ", jev.ask({"message": vague}, {"team": QUESTIONS["team"]})["answers"]["team"]["probabilities"])
    print("Without other:", jev.ask({"message": vague}, no_fallback)["answers"]["team"]["probabilities"])
else:
    print("Live only. In replay mode, read the saved answer above: other won because it was available.")

## Takeaways

1. The request is **state + typed questions**; answers come back under the same IDs.
2. Confidence and score are simple functions of the probabilities: recompute them, don't trust them blindly.
3. A fallback option and a review path are part of the design, not an afterthought.